# Baseline Model
Building a simple baseline model to understand how well basic machine learning works on our processed dataset.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, 
    confusion_matrix, 
    precision_score, 
    recall_score, 
    f1_score, 
    roc_auc_score, 
    average_precision_score
)

# Load the processed data
df = pd.read_csv('../data/processed.csv')
print(f"Data loaded. Shape: {df.shape}")

## 1. Prepare Data and Split
- **Target (y):** We want to predict `isFraud`.
- **Features (X):** Everything else. We drop `step` because we already extracted the `hour` feature from it.
- **Train/Test Split:** We split the data so the model learns on 80% (train) and takes a "final exam" on the remaining 20% (test). We use `stratify=y` to ensure both groups have the exact same percentage of fraud cases.

In [ ]:
X = df.drop(columns=['isFraud', 'step'])
y = df['isFraud']

# Split the data 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Fraud cases in train: {y_train.sum()}")
print(f"Fraud cases in test: {y_test.sum()}")

## 2. The Accuracy Trap (Dummy Model)
Why don't we just look at "Accuracy"? In fraud detection, 99% of transactions are normal. If a model just guesses "Not Fraud" every single time without even looking at the data, it will be 99% accurate! But it will catch exactly zero fraudsters. We create a Dummy Model to prove this point.

In [ ]:
# A dummy model that predicts 0 (Not Fraud) for everything in the test set
dummy_preds = [0] * len(y_test)
dummy_accuracy = accuracy_score(y_test, dummy_preds) * 100

print(f"Dummy Model Accuracy (Always guesses 'Not Fraud'): {dummy_accuracy:.4f}%")
print(f"Fraud caught by Dummy Model: {sum([p == 1 and a == 1 for p, a in zip(dummy_preds, y_test)])}")

## 3. Train Baseline Model (Logistic Regression)
We will use Logistic Regression as our baseline. It's a simple, fast algorithm. 
- **StandardScaler:** Logistic Regression struggles if some numbers are huge (like bank balances in the millions) and others are small (like hour, which is 0-23). The scaler shrinks everything to a standard size so the model looks at them fairly.
- **class_weight='balanced':** Because fraud is so rare, the model might just ignore it. This setting tells the model to pay extra attention to fraud and penalizes it heavily when it makes a mistake on a fraud case.

In [ ]:
# Create a pipeline that scales the data first, then trains the model
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('log_reg', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])

# Train the model
pipeline.fit(X_train, y_train)
print("Baseline model trained!")

## 4. Evaluate the Baseline Model
Since accuracy is a trap for fraud detection, we use these metrics instead:
- **Confusion Matrix:** Shows a grid of True Positives, True Negatives, False Positives (false alarms), and False Negatives (missed fraud).
- **Precision:** When the model raises a "Fraud!" alarm, how often is it actually a real fraudster? (High precision = fewer angry customers blocked from their money).
- **Recall:** Out of all the actual fraud in the world, what percentage did we successfully catch? (High recall = less money lost to criminals).
- **F1 Score:** A balanced average of Precision and Recall. Useful if you want one single number to summarize both.
- **ROC-AUC:** Measures the model's ability to rank fraud above non-fraud. A score of 1.0 is perfect, 0.5 is random guessing.
- **PR-AUC (Average Precision):** Similar to ROC-AUC, but specifically focused on the rare class (fraud). It measures how well the model avoids false alarms while trying to catch as much fraud as possible.

In [ ]:
# Get predictions (0 or 1)
y_pred = pipeline.predict(X_test)

# Get probabilities (percentages) for the ROC and PR metrics
y_prob = pipeline.predict_proba(X_test)[:, 1]

print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))
print("\n--- Metrics ---")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob):.4f}")
print(f"PR-AUC:    {average_precision_score(y_test, y_prob):.4f}")